<a href="https://colab.research.google.com/github/linlih/ML_By_NTU_Hung-yi-Lee/blob/master/ml_2019spring_hw1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## HW1????

????:??9?????????????,?????????PM2.5??

## ????

Kaggle???????????????,???????kaggle.json???

kaggle.json?????Kaggle?????????????????:https://www.kaggle.com/???/account ,?????????????,?????API?????Create New API Token????????

??:??????????,????,???????????

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!ls ~/.kaggle

In [ ]:
!kaggle competitions download -c ml2019spring-hw1

In [ ]:
!ls

### ????

?Kaggle?????????train.csv????????,??Linux???iconv?????UTF-8?????,?????,??Notepad++??????UTF-8???

In [ ]:
import pandas

# data = pandas.read_csv('train.csv', encoding='utf-8') 
# ?????????????big5
data = pandas.read_csv('train.csv', encoding='big5')
data.head()

In [ ]:
data.info() # ?????????????non-null,????????float

## ??????

????????:
https://www.bilibili.com/video/av10590361/?p=9

????????????

???????24??????,???12??,???20?,??????18?????,????????PM2.5?????,??9????PM2.5??????10????PM2.5?????????????

??24???,??????????15(24 - 9)?example????240?,???????example??3600(240 X 15)??

In [ ]:
import numpy as np
np.set_printoptions(suppress=True) # ???????????

data_ = data.drop(['??','??', '??'], axis=1)

x_train_list = []
y_train_list = []
data_ = data_.replace(['NR'], [0.0])
data_ = np.array(data_).astype(float)

# ????????????
for i in range(0, 4320, 18): # ???12*20*18=4320????,?18???
    for j in range(15): # ??????15???example
        example = data_[i:i+18, j:j+9] # example shape (18,9),???????18?????
        label = data_[i+9, j+9] # PM2.5???
        x_train_list.append(example)
        y_train_list.append(label)
x_train_all = np.array(x_train_list)
y_train = np.array(y_train_list)       

In [ ]:
x_train = []
for i in range(3600): # ?3600?examples
    x_train.append([])
    for j in range(9):
        x_train[i].append(x_train_all[i,9,j])
x_train = np.array(x_train)

In [ ]:
print(x_train.shape)
print(y_train.shape)

## ????

????????,???9????PM2.5????,????????:y=wx+b,??????w?b???????

1. function set

$$
y=b+w_{0} x_{0}+w_{1} x_{1}+\ldots+w_{9} x_{9}
 $$

2. Loss function
 $$ 
L=\frac{1}{2*m} \Sigma\left(y-y_{\text {train}}\right)^{2}
 $$

3. Find best function : Compute Gradient
$$ 
b=b-\eta * \frac{\vartheta L}{\vartheta b}=b-\eta * \Sigma \frac{1}{m} *\left(y-y_{\text {train}}\right) \frac{\vartheta y}{\vartheta b}=b-\eta * \Sigma \frac{1}{m} *\left(y-y_{\text {train}}\right)
 $$
 $$ 
w_{i}=w_{i}-\eta * \frac{\vartheta L}{\vartheta w_{i}}=w_{i}-\eta * \Sigma \frac{1}{m}\left(y-y_{\text {train}}\right) * x_{i}
 $$
 
 ??m?????

1. Gradient Descent??

In [ ]:
# Gradient Descent
def GD(x_train, y_train, iteration, lr):
    b = 0.0 # ??????
    w = np.ones(9) # ?????

    loss_store = []
    for j in range(iteration):
        w_grad_sum = 0.0
        b_grad_sum = 0.0
        loss = 0.0
        # ??????????,?????????????
        for i in range(y_train.shape[0]):
            y = b + np.dot(w, x_train[i])
            w_grad = -1.0/y_train.shape[0]*(y_train[i] - y)*x_train[i]
            b_grad = -1.0/y_train.shape[0]*(y_train[i] - y)
            
            w_grad_sum += w_grad
            b_grad_sum += b_grad
            
            error = y_train[i] - y
            loss += error**2
        w = w - lr*w_grad_sum
        b = b - lr*b_grad_sum
        loss = loss/y_train.shape[0]
        print("after {} epochs, the loss is: ".format(j),loss)
        loss_store.append(loss)

    return w, b, loss_store

In [ ]:
w, b, loss_store = GD(x_train, y_train, 1000, 0.000001)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(np.arange(len(loss_store)),loss_store)

2. Stochastic Gradient Descent ??

In [ ]:
# Stochastic Gradient Descent
def SGD(x_train, y_train, iteration, lr):
    b = 0.0 # ??????
    w = np.ones(9) # ?????

    loss_store = []
    
    for j in range(iteration):
        # ??????????????Gradient
        rand = np.random.randint(0, y_train.shape[0])
    
        y = b + np.dot(w, x_train[rand])
        w_grad = -1.0*(y_train[rand] - y)*x_train[rand]
        b_grad = -1.0*(y_train[rand] - y)

        error = y_train[rand] - y
        loss = error**2
        w = w - lr*w_grad
        b = b - lr*b_grad
        
        loss_store.append(loss)

    return w, b, loss_store

In [ ]:
w, b, loss_store = SGD(train_x, train_y, 50000, 0.0000001)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(np.arange(len(loss_store)),loss_store)

3. Adagrad??

In [ ]:
def Adagrad(x_train, y_train, iteration, lr):
    b = 0.0 # ??????
    w = np.ones(9) # ?????
    b_lr = 0.0
    w_lr = 0.0

    loss_store = []
    for j in range(iteration):
        w_grad_sum = 0.0
        b_grad_sum = 0.0
        loss = 0.0
        # ??????????,?????????????
        for i in range(y_train.shape[0]):
            y = b + np.dot(w, x_train[i])
            w_grad = -1.0/y_train.shape[0]*(y_train[i] - y)*x_train[i]
            b_grad = -1.0/y_train.shape[0]*(y_train[i] - y)
            
            w_grad_sum += w_grad
            b_grad_sum += b_grad
       
            error = y_train[i] - y
            
            loss += error**2
        
        w_lr = w_lr + w_grad_sum**2
        b_lr = b_lr + b_grad_sum**2
        
        w = w - lr/np.sqrt(w_lr) * w_grad_sum
        b = b - lr/np.sqrt(b_lr) * b_grad_sum
        loss = loss/y_train.shape[0]/9
        print("after {} epochs, the loss is: ".format(j),loss)
        loss_store.append(loss)

    return w, b, loss_store

In [ ]:
w, b, loss_store = Adagrad(x_train, y_train, 100, 0.1)
# ??????(10000, 0.1): after 9999 epochs, the loss is:  5.13767154668965
# ??????(5000 , 0.5): after 4999 epochs, the loss is:  4.795303444341576
# ??????(5000 , 1.0): after 4999 epochs, the loss is:  4.794390440080361

In [ ]:
import matplotlib.pyplot as plt
plt.plot(np.arange(len(loss_store)),loss_store)

## ????

### ???????

In [ ]:
import pandas
test_data = pandas.read_csv('test.csv', encoding='utf-8', header=None)# ?????????????
test_data.head()

In [ ]:
test_data_ = test_data.drop([0, 1], axis=1)

test_data_ = test_data_.replace(['NR'], [0.0])
test_data_ = np.array(test_data_).astype(float)

test_data_x_list = []

# ????????????
for i in range(0, 4320, 18): # ??????4320????,?18???
    example = test_data_[i:i+18, 0:9] # example shape (18,9)
    test_data_x_list.append(example)

In [ ]:
test_data_x_arr = np.array(test_data_x_list)

test_data_x = []
for i in range(240): # ??240?????
    test_data_x.append([])
    for j in range(9):
        test_data_x[i].append(test_data_x_arr[i,9,j])

In [ ]:
np.array(test_data_x).shape

In [ ]:
# ?????????
predict = []
for i in range(240):
    y = b + np.dot(w,test_data_x[i])
    predict.append(y)

results = pd.Series(predict, name="value")
submission = pd.concat([pd.Series(["id_%d" % i for i in range(240)],name = "id"),results],axis = 1)
submission.to_csv('submission.csv', index=False)

In [ ]:
# ??????
res = []
for i in range(240):
    res.append(int(predict[i]))
res

results = pd.Series(res, name="value")
submission = pd.concat([pd.Series(["id_%d" % i for i in range(240)],name = "id"),results],axis = 1)
submission.to_csv('submission.csv', index=False)

In [ ]:
submission.head()

## ????

??Adagrad????10000?,????0.1,???????????:

w: [ 0.0480092  -0.00509123  0.01689356 -0.02796585  0.04653025  0.13340423 -0.31917557  0.1099347   0.96548574]

b: 0.7212832358712717

???Kaggle???????:6.50304,?private Leaderboard?13????

?????????????:6.48652

----

??Adagrad????5000?,????0.5,???????????:

w: [ 0.00667528 -0.0367696   0.18275847 -0.19846756 -0.0345893   0.44443787
 -0.53391454  0.01922113  1.07311218]
 
b: 2.1087492095813527

???Kaggle???????:5.95125,?private Leaderboard?6????

??????????:5.98400

----

??Adagrad????5000?,????1.0,???????????:

w: [ 0.0071957  -0.04571828  0.19905271 -0.20752237 -0.04349567  0.46194178
 -0.54319042  0.0157119   1.07702657]
 
b: 2.152498260439194

???Kaggle???????:5.93645,?private Leaderboard?6????




In [ ]:
!kaggle competitions submit ml2019spring-hw1 -f submission.csv -m "submit from colab"

## ????

??????????????zhexi??,?????????!

??? : https://www.cnblogs.com/HL-space/p/10676637.html

??? : https://zhuanlan.zhihu.com/p/65400308

maplezzz : https://github.com/maplezzz/NTU_ML2017_Hung-yi-Lee_HW/tree/master/HW1
